# Teks — TextCNN / BiLSTM + attention dari nol (tanpa pretrained), k-fold + OOF
**Kapan dipakai:** (1) model tambahan yang sifat salahnya beda dari BERT/TF-IDF untuk ensemble,
(2) bahasa/kosakata aneh yang tidak dikenal model pretrained (kode, slang berat, log, DNA,
urutan ID), (3) GPU lemah / waktu mepet: satu epoch hitungan detik. Juri sering menghargai
arsitektur yang dijelaskan dari dasar.

**Isi:** tokenisasi kata sederhana → kosakata dari train → `TextCNN` (Kim 2014: konvolusi
lebar 2-5 + max-pool) atau `BiLSTMAttention` (LSTM dua arah + pooling attention). Pilih di
`make_model`.

**Saat lomba yang diubah:** sel Konfigurasi dan baris `return` di `make_model`.

**Data demo:** SmSA (`sentiment`, 3 kelas).

In [ ]:
import html
import os
import re
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from torch import nn
from torch.nn.functional import cross_entropy, softmax

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"                      # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/textcnn_bilstm"
OOF_FOLDER = "../../../outputs/text/oof"
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"     # GANTI saat lomba: file panitia
SUBMISSION_NAME = "submission.csv"
ANSWER_CSV = f"{DATA_FOLDER}/smsa_test_labels.csv"             # HAPUS saat lomba (kunci demo)
ID_COL = "id"                    # GANTI saat lomba
TEXT_COL = "text"                # GANTI saat lomba
TARGET = "label"                 # GANTI saat lomba
SUBMIT_COL = TARGET
N_FOLDS = 5
SPLIT_SEED = 42
NOTEBOOK = "textcnn_bilstm"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
MODEL_NAME = "textcnn"           # GANTI: "bilstm_attn" bila make_model diganti
VOCAB_SIZE = 30000               # kata terbanyak yang disimpan
MIN_FREQ = 2                     # kata muncul < 2 kali jadi <unk>
MAX_TOKENS = 200                 # kata per teks (lebih panjang dipotong)
EMBED_DIM = 200
N_FILTERS = 128                  # TextCNN: filter per lebar
KERNEL_SIZES = [2, 3, 4, 5]      # TextCNN: lebar n-gram
HIDDEN = 128                     # BiLSTM: ukuran tiap arah
DROPOUT = 0.4
EPOCHS = 8
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
SEED = 42

## Metrik

In [ ]:
METRIC_NAME = "f1_macro"         # GANTI saat lomba, ikut baris return di bawah


def score(truth, predicted):
    return f1_score(truth, predicted, average="macro")


print("metrik:", METRIC_NAME, "| device:", DEVICE)

## Baca data
Teks kosong/NaN jadi string kosong, lalu `clean_basic` (html + spasi rapi). Label diubah ke indeks
0..K-1 urut `np.unique` (format OOF bersama).

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)


def clean_basic(text):
    return " ".join(html.unescape(text).split())


train_text = train[TEXT_COL].fillna("").astype(str)
test_text = test[TEXT_COL].fillna("").astype(str)
train_text = train_text.apply(clean_basic)
test_text = test_text.apply(clean_basic)
classes = np.unique(train[TARGET])
y = np.searchsorted(classes, train[TARGET].to_numpy())
print("train:", train.shape, "| test:", test.shape, "| NaN target:", train[TARGET].isna().sum())
print("kelas:", list(classes), "| jumlah:", np.bincount(y).tolist())
train[[ID_COL, TEXT_COL, TARGET]].head()

## Buat sample_submission demo (HAPUS saat lomba)
Data latihan tidak punya `sample_submission`. Saat lomba file panitia dipakai (`SAMPLE_CSV`);
sel ini tidak menimpa file di luar `OUTPUT_FOLDER`.

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUBMIT_COL] = train[TARGET].iloc[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    demo_sample.to_csv(SAMPLE_CSV, index=False)
    print("sample demo ditulis ke", SAMPLE_CSV)
else:
    print("SAMPLE_CSV bukan file demo: dibiarkan apa adanya")

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
print("sample_submission:", sample.shape, sample.columns.tolist())
print("id test = id sample:", set(test[ID_COL]) == set(sample[ID_COL]))
sample.head(3)

## Split (k-fold)
Aturan format OOF: `StratifiedKFold` acak seed `SPLIT_SEED` pada urutan baris train apa adanya,
jadi OOF notebook ini sefold dengan `finetune_oof`, `baseline_oof`, dll. dan bisa di-ensemble.

In [ ]:
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
folds = list(splitter.split(train_text, y))
val_sizes = []
for _, val_index in folds:
    val_sizes.append(len(val_index))
print("fold:", len(folds), "| ukuran validasi:", val_sizes)

## Kosakata dan ID kata
Kata = deret huruf/angka (`\w+`), huruf kecil. Kosakata dari teks train (tanpa label, jadi tidak
bocor). Alternatif: tambah teks test ke `counts` (transduktif, kata khas test ikut dikenal).

In [ ]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())


counts = Counter()
for text in train_text:
    counts.update(tokenize(text))
# for text in test_text:          # transduktif: kosakata juga dari test
#     counts.update(tokenize(text))
vocab = {"<pad>": 0, "<unk>": 1}
for word, count in counts.most_common(VOCAB_SIZE):
    if count >= MIN_FREQ:
        vocab[word] = len(vocab)
print("kosakata:", len(vocab), "| contoh:", tokenize(train_text.iloc[0])[:10])

In [ ]:
def to_ids(texts):
    matrix = np.zeros((len(texts), MAX_TOKENS), dtype="int64")
    for row, text in enumerate(texts):
        ids = []
        for word in tokenize(text)[:MAX_TOKENS]:
            ids.append(vocab.get(word, 1))
        matrix[row, :len(ids)] = ids
    return matrix


train_ids = to_ids(train_text)
test_ids = to_ids(test_text)
token_counts = (train_ids != 0).sum(axis=1)
unknown_share = (train_ids == 1).sum() / max(1, token_counts.sum())
print("kata per teks p50/p95:", np.percentile(token_counts, [50, 95]), "| <unk>:",
      f"{unknown_share:.1%}")

## Model
`TextCNN`: tiap lebar konvolusi menangkap n-gram, max-pool mengambil sinyal terkuat di mana pun.
`BiLSTMAttention`: LSTM membaca dua arah, attention memberi bobot tiap kata (padding diabaikan).

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, n_classes):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.convs = nn.ModuleList()
        for width in KERNEL_SIZES:
            self.convs.append(nn.Conv1d(EMBED_DIM, N_FILTERS, width, padding=width // 2))
        self.drop = nn.Dropout(DROPOUT)
        self.out = nn.Linear(N_FILTERS * len(KERNEL_SIZES), n_classes)

    def forward(self, ids):
        hidden = self.embed(ids).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            pooled.append(torch.relu(conv(hidden)).amax(dim=2))
        return self.out(self.drop(torch.cat(pooled, dim=1)))


print("TextCNN siap")

In [ ]:
class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, n_classes):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(HIDDEN * 2, 1)
        self.drop = nn.Dropout(DROPOUT)
        self.out = nn.Linear(HIDDEN * 2, n_classes)

    def forward(self, ids):
        hidden, _ = self.lstm(self.drop(self.embed(ids)))
        weights = self.attn(hidden).squeeze(-1)
        weights = weights.masked_fill(ids == 0, -1e4)
        weights = torch.softmax(weights, dim=1).unsqueeze(-1)
        return self.out(self.drop((hidden * weights).sum(dim=1)))


def make_model():
    return TextCNN(len(vocab), len(classes))
    # return BiLSTMAttention(len(vocab), len(classes))   # urutan kata penting, teks panjang


print("model:", type(make_model()).__name__)

## Latih dan prediksi
Tiap batch dipotong selebar teks terpanjangnya (lebih cepat). Bobot terbaik per epoch dipilih
dari skor validasi fold.

In [ ]:
def to_batch(ids):
    width = max(5, int((ids != 0).sum(axis=1).max()))    # min 5: konvolusi lebar 5
    return torch.tensor(ids[:, :width]).to(DEVICE)


def predict_proba(model, ids):
    model.eval()
    parts = []
    with torch.no_grad():
        for start in range(0, len(ids), BATCH_SIZE * 4):
            logits = model(to_batch(ids[start:start + BATCH_SIZE * 4]))
            parts.append(softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(parts)


def train_epoch(model, optimizer, fit_index, rng):
    model.train()
    order = rng.permutation(fit_index)
    for start in range(0, len(order), BATCH_SIZE):
        chosen = order[start:start + BATCH_SIZE]
        targets = torch.tensor(y[chosen]).to(DEVICE)
        loss = cross_entropy(model(to_batch(train_ids[chosen])), targets)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()


print("train_epoch / predict siap")

In [ ]:
def snapshot(model):
    state = {}
    for key, tensor in model.state_dict().items():
        state[key] = tensor.detach().clone()
    return state


def train_fold(fit_index, val_index, rng):
    torch.manual_seed(SEED)
    model = make_model().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best, state = -1.0, None
    for epoch in range(EPOCHS):
        train_epoch(model, optimizer, fit_index, rng)
        value = score(y[val_index], predict_proba(model, train_ids[val_index]).argmax(axis=1))
        if value > best:
            best, state = value, snapshot(model)
    model.load_state_dict(state)
    print(f"  {METRIC_NAME} validasi terbaik {best:.4f}")
    return predict_proba(model, train_ids[val_index]), predict_proba(model, test_ids)


print("train_fold siap")

## Latih k-fold

In [ ]:
oof = np.zeros((len(train), len(classes)), dtype="float32")
test_proba = np.zeros((len(test), len(classes)), dtype="float32")
rng = np.random.default_rng(SEED)
began = time.time()
for number, (fit_index, val_index) in enumerate(folds):
    val_proba, test_part = train_fold(fit_index, val_index, rng)
    oof[val_index] = val_proba
    test_proba += test_part / len(folds)
    print(f"fold {number}: {time.time() - began:.0f} dtk")
total_seconds = round(time.time() - began, 1)
note = (f"{type(make_model()).__name__} | vocab {len(vocab)} len {MAX_TOKENS} emb {EMBED_DIM} "
        f"epoch {EPOCHS} lr {LEARNING_RATE}")
print(note)

## Evaluasi OOF
Skor dihitung di baris train yang punya prediksi OOF (semua baris bila semua fold jalan).

In [ ]:
filled = oof.sum(axis=1) > 0
oof_pred = oof.argmax(axis=1)
oof_score = score(y[filled], oof_pred[filled])
print(f"{METRIC_NAME} OOF ({filled.sum()} baris): {oof_score:.4f}")
print(classification_report(y[filled], oof_pred[filled], target_names=classes.astype(str)))

## Simpan OOF dan catat percobaan
`oof_<MODEL_NAME>.npy` / `test_<MODEL_NAME>.npy` hanya bila semua baris train terisi, untuk
`ensemble_oof`. `percobaan.csv` bertambah satu baris tiap run.

In [ ]:
if filled.all():
    np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_proba)
    print("OOF dan test disimpan:", MODEL_NAME)
else:
    print("OOF tidak lengkap (FOLDS_TO_RUN?): tidak disimpan")
row = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": NOTEBOOK, "model": MODEL_NAME,
       "metrik": METRIC_NAME, "skor_cv": round(oof_score, 4), "detik": total_seconds,
       "catatan": note}
log_path = f"{OOF_FOLDER}/percobaan.csv"
pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
history = pd.read_csv(log_path)
print(history.tail(5).to_string())

## Submission
Kolom id diambil dari `sample_submission`; prediksi dipetakan lewat id.

In [ ]:
test_pred = test_proba.argmax(axis=1)
pred_by_id = pd.Series(classes[test_pred], index=test[ID_COL])

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUBMIT_COL] = sample[ID_COL].map(pred_by_id)
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
predicted_share = submission[SUBMIT_COL].value_counts(normalize=True)
print("sebaran prediksi test:", predicted_share.round(3).to_dict())
submission.head()

## Cek submission terhadap sample_submission
Salinan validator `arsenal/umum/simpan_muat` #15 dan #16: hanya membaca dan mencetak. Hasil
akhir harus `MASALAH: 0`.

In [ ]:
SUBMISSION_CSV = f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}"
ID_COLUMN = ID_COL

sample = pd.read_csv(SAMPLE_CSV)
written = pd.read_csv(SUBMISSION_CSV)
problems = []


def report(failed, text):
    if failed:
        problems.append(text)


wanted = sample[ID_COLUMN]
ids = written.get(ID_COLUMN, pd.Series(dtype=wanted.dtype))
lost = set(wanted) - set(ids)
extra = set(ids) - set(wanted)
kept = ids[ids.isin(wanted)].drop_duplicates().tolist()
expected = wanted[wanted.isin(ids)].tolist()
report(written.columns.tolist() != sample.columns.tolist(), "kolom / urutan kolom beda")
report(len(written) != len(sample), f"jumlah baris {len(written)}, sample {len(sample)}")
report(lost, f"id hilang {len(lost)}, mis. {sorted(lost)[:3]}")
report(extra, f"id asing {len(extra)}, mis. {sorted(extra)[:3]}")
report(ids.duplicated().any(), f"id kembar: {ids.duplicated().sum()} baris")
report(kept != expected, "urutan id beda dari sample")

In [ ]:
LABEL_COLUMNS = [SUBMIT_COL]     # kolom isi kelas ([] = lewati cek label)
ALLOWED = list(classes)          # label sah = label train, [] = lewati cek

for column in sample.columns:
    if column in written.columns:
        empty = written[column].isna().sum()
        report(empty, f"{column}: {empty} nilai kosong")
        report(written[column].dtype != sample[column].dtype,
               f"{column}: tipe {written[column].dtype}, sample {sample[column].dtype}")
for column in LABEL_COLUMNS:
    if ALLOWED and column in written.columns:
        foreign = set(written[column].dropna()) - set(ALLOWED)
        report(foreign, f"{column}: label di luar ALLOWED {sorted(foreign)[:5]}")
        print(written[column].value_counts())   # semua satu kelas -> curigai model
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)
Hanya untuk data latihan yang punya kunci (`ANSWER_CSV`). Hapus sel ini saat lomba.

In [ ]:
answers = pd.read_csv(ANSWER_CSV)
truth_by_id = pd.Series(answers[TARGET].to_numpy(), index=answers[ID_COL])
truth_index = np.searchsorted(classes, truth_by_id.reindex(submission[ID_COL]).to_numpy())
pred_index = np.searchsorted(classes, submission[SUBMIT_COL].to_numpy())
print(f"{METRIC_NAME} di test (kunci): {score(truth_index, pred_index):.4f}")
print(f"{METRIC_NAME} OOF train: {oof_score:.4f}")